In [1]:
import os
import os, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Sequential
import torch.optim as optim
import voxelmorph as vxm
import neurite as ne
import scipy.ndimage

os.environ['VXM_BACKEND'] = 'pytorch'

backend:pytorch
Pytorch


In [2]:
os.environ.get('VXM_BACKEND')

'pytorch'

In [3]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [4]:
# 画像を読み込み
x_train = np.load('Data/TrainData_NoBed.npz')['Train']
x_train = np.transpose(x_train, (3, 0, 1, 2))

print('Resized train vol_shape:', x_train.shape[1:])
print('Resized train shape:', x_train.shape)

Resized train vol_shape: (128, 256, 256)
Resized train shape: (400, 128, 256, 256)


In [5]:
import torch

def vxm_data_generator(x_data, batch_size):
    vol_shape = x_data.shape[1:]  # データ形状を取得
    ndims = len(vol_shape)
    
    zero_phi = np.zeros([batch_size, *vol_shape, ndims])
    
    while True:
        idx1 = np.random.randint(0, x_data.shape[0], size=batch_size)
        moving_images = x_data[idx1, ..., np.newaxis]
        # ファインチューニングでは同じ症例同士のペアを避ける
        idx2 = np.random.randint(0, x_data.shape[0], size=batch_size)
        while np.any(idx2 == idx1):
            same_case = idx2 == idx1
            idx2[same_case] = np.random.randint(0, x_data.shape[0], size=same_case.sum())
        fixed_images = x_data[idx2, ..., np.newaxis]

        # TensorFlowからPyTorchのデータ形式に変換
        moving_images = torch.tensor(moving_images).permute(0, 4, 1, 2, 3).float()
        fixed_images = torch.tensor(fixed_images).permute(0, 4, 1, 2, 3).float()

        # チャンネルを最初の次元に追加
        moving_images = moving_images.permute(0, 1, 2, 3, 4)  # チャンネルを最初の次元に移動
        fixed_images = fixed_images.permute(0, 1, 2, 3, 4)  # チャンネルを最初の次元に移動

        inputs = [moving_images, fixed_images]
        outputs = [fixed_images, zero_phi]

        yield (inputs, outputs)

In [6]:
train_generator = vxm_data_generator(x_train, batch_size=2)
in_sample, out_sample = next(train_generator)

# in_sampleとout_sampleの内容を確認する
print("Input Sample Shapes:")
print("Moving Images Shape:", in_sample[0].shape)
print("Fixed Images Shape:", in_sample[1].shape)

print("\nOutput Sample Shapes:")
print("Moved Images (Fixed) Shape:", out_sample[0].shape)
print("Zero Gradient Shape:", out_sample[1].shape)

Input Sample Shapes:
Moving Images Shape: torch.Size([2, 1, 128, 256, 256])
Fixed Images Shape: torch.Size([2, 1, 128, 256, 256])

Output Sample Shapes:
Moved Images (Fixed) Shape: torch.Size([2, 1, 128, 256, 256])
Zero Gradient Shape: (2, 128, 256, 256, 3)


In [7]:
mse_loss = vxm.losses.MSE().loss
grad_loss = vxm.losses.Grad('l2').loss

def total_loss(y_true, y_pred):
    mse = mse_loss(y_true, y_pred)
    grad = grad_loss(y_true, y_pred)
    return mse + 0.01 * grad, mse, grad
#     return mse_loss(y_true, y_pred)

def MSE_Loss(y_true, y_pred):
    y_true = y_true.to(device)
    y_pred = y_pred.to(device)
    mse = mse_loss(y_true, y_pred)
    return mse

def lncc_loss(I, J, window=9, eps=1e-5):
    # I, J: (B, 1, D, H, W)
    padding = window // 2
    weight = torch.ones(1, 1, window, window, window, device=I.device)

    I2 = I * I
    J2 = J * J
    IJ = I * J

    I_sum = F.conv3d(I, weight, padding=padding)
    J_sum = F.conv3d(J, weight, padding=padding)
    I2_sum = F.conv3d(I2, weight, padding=padding)
    J2_sum = F.conv3d(J2, weight, padding=padding)
    IJ_sum = F.conv3d(IJ, weight, padding=padding)

    win_size = window ** 3
    u_I = I_sum / win_size
    u_J = J_sum / win_size

    cross = IJ_sum - u_J * I_sum - u_I * J_sum + u_I * u_J * win_size
    I_var = I2_sum - 2 * u_I * I_sum + u_I * u_I * win_size
    J_var = J2_sum - 2 * u_J * J_sum + u_J * u_J * win_size

    lncc = cross * cross / (I_var * J_var + eps)
    return -torch.mean(lncc)  # maximize LNCC → minimize -LNCC

In [8]:
# configure unet input shape (concatenation of moving and fixed images)
ndim = 3
unet_input_features = 2
# inshape = (*x_train.shape[1:], unet_input_features)

nb_features = [
    [32, 64, 64, 64, 64],
    [64, 64, 64, 64, 64, 32, 16, 16]
]


In [9]:
import voxelmorph as vxm
import inspect

print(vxm.__file__)
print(vxm.networks.__file__)
print([name for name in dir(vxm.networks) if "VxmDense" in name])

C:\Users\user\anaconda3\envs\nn\lib\site-packages\voxelmorph\__init__.py
C:\Users\user\anaconda3\envs\nn\lib\site-packages\voxelmorph\torch\networks.py
['VxmDense', 'VxmDense1', 'VxmDense2', 'VxmDense_128_256', 'VxmDense_128_256_256']


In [10]:
model3D = vxm.networks.VxmDense_128_256_256((128, 256, 256), nb_features, int_steps=0)
model3D.to(device)
optimizer = optim.Adam(model3D.parameters(), lr=1e-4)

transformer = vxm.layers.SpatialTransformer((64, 128, 128)).to(device)
transformer256 = vxm.layers.SpatialTransformer((128, 256, 256)).to(device)

[64, 128, 128]


C:\Users\user\anaconda3\envs\nn\lib\site-packages\torch\functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\TensorShape.cpp:3550.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [38]:
import math
from pathlib import Path
import torch
import matplotlib.pyplot as plt

band_names = ['LLL', 'LLH', 'LHL', 'LHH', 'HLL', 'HLH', 'HHL', 'HHH']

wavelet_vis_enabled = False
wavelet_vis_every = 100
wavelet_vis_dir = Path('wavelet_stage_outputs')
wavelet_vis_dir.mkdir(exist_ok=True)

class Haar3DAnalysisOnly(nn.Module):
    def __init__(self):
        super().__init__()

        hL = torch.tensor([1.0, 1.0], dtype=torch.float32) / math.sqrt(2.0)
        hH = torch.tensor([1.0, -1.0], dtype=torch.float32) / math.sqrt(2.0)

        filters = []
        names = []

        for z_name, z_filter in zip(['L', 'H'], [hL, hH]):
            for y_name, y_filter in zip(['L', 'H'], [hL, hH]):
                for x_name, x_filter in zip(['L', 'H'], [hL, hH]):
                    kernel = (
                        z_filter[:, None, None]
                        * y_filter[None, :, None]
                        * x_filter[None, None, :]
                    )
                    filters.append(kernel)
                    names.append(z_name + y_name + x_name)

        weight = torch.stack(filters, dim=0).unsqueeze(1)
        self.register_buffer('weight', weight)
        self.names = names

    def forward(self, x):
        x = F.pad(x, (0, 1, 0, 1, 0, 1))
        return F.conv3d(x, self.weight, stride=1, padding=0)

def analysis_filter_3d(x, analysis_layer):
    return analysis_layer(x)

def down_sampling_3d(w):
    return w[:, :, ::2, ::2, ::2]

def up_sampling_3d(w_down):
    B, C, D, H, W = w_down.shape
    w_up = torch.zeros(
        B, C, D * 2, H * 2, W * 2,
        dtype=w_down.dtype,
        device=w_down.device
    )
    w_up[:, :, ::2, ::2, ::2] = w_down
    return w_up

def make_3d_filter(fz, fy, fx):
    return fz[:, None, None] * fy[None, :, None] * fx[None, None, :]

def create_synthesis_filters(device):
    low = torch.tensor([1.0, 1.0], dtype=torch.float32, device=device) / math.sqrt(2.0)
    high = torch.tensor([1.0, -1.0], dtype=torch.float32, device=device) / math.sqrt(2.0)

    filters = torch.stack([
        make_3d_filter(low, low, low),
        make_3d_filter(low, low, high),
        make_3d_filter(low, high, low),
        make_3d_filter(low, high, high),
        make_3d_filter(high, low, low),
        make_3d_filter(high, low, high),
        make_3d_filter(high, high, low),
        make_3d_filter(high, high, high),
    ], dim=0)

    filters = torch.flip(filters, dims=[1, 2, 3]).unsqueeze(1)
    return filters

def synthesis_filter_3d(w_up, synthesis_filters):
    B, C, D, H, W = w_up.shape
    filtered_bands = []

    for i in range(C):
        band = w_up[:, i:i + 1, :, :, :]
        kernel = synthesis_filters[i:i + 1]
        filtered = F.conv3d(band, kernel, stride=1, padding=1)
        filtered = filtered[:, :, :D, :H, :W]
        filtered_bands.append(filtered)

    filtered_bands = torch.cat(filtered_bands, dim=1)
    reconstructed = torch.sum(filtered_bands, dim=1, keepdim=True)
    return reconstructed, filtered_bands

analysis = Haar3DAnalysisOnly().to(device)
synthesis_filters = create_synthesis_filters(device)
analysis_names = analysis.names

In [ ]:
# 80k epoch pretraining is already complete for this workflow.
# Keep this False to skip the expensive synthetic-deformation pretraining cell.
run_pretraining = False
if not run_pretraining:
    print('Pretraining is skipped. Run the next cell to fine-tune from the 80k checkpoint.')


## Different-patient, lung-field-preferred fine-tuning

This cell loads the completed 80k checkpoint and fine-tunes it on distinct patients sampled from `Data/TrainData_NoBed.npz`. It prefers a masked archive or a mask key when present; otherwise it prints a clear warning and runs on unmasked volumes.


In [ ]:
# Fine-tuning: different patients from TrainData_NoBed, preferring lung-masked data.
# This cell assumes that cells defining vxm, the wavelet pipeline, nb_features,
# analysis, synthesis_filters, MSE_Loss, and device have already been run.
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt

DATA_DIR = Path('Data')
RAW_DATA_PATH = DATA_DIR / 'TrainData_NoBed.npz'
PRETRAINED_MODEL_PATH = Path('model_analysis_pipeline_pretrain.pth')

# Prefer a pre-masked volume archive if one is supplied. If none is found, this
# code looks for a lung-mask key in TrainData_NoBed.npz and masks the volumes.
MASKED_ARCHIVE_CANDIDATES = [
    DATA_DIR / 'TrainData_NoBed_LungMasked.npz',
    DATA_DIR / 'TrainData_NoBed_lung_masked.npz',
    DATA_DIR / 'TrainData_NoBed_Masked.npz',
    DATA_DIR / 'TrainData_NoBed_masked.npz',
]
VOLUME_KEYS = ('Train_lung_masked', 'train_lung_masked', 'TrainMasked', 'masked_train', 'Train')
MASK_KEYS = ('Train_lung_mask', 'train_lung_mask', 'LungMask', 'lung_mask', 'TrainMask', 'train_mask', 'Mask', 'mask')

if not PRETRAINED_MODEL_PATH.exists():
    raise FileNotFoundError(
        f'80k pretrained checkpoint was not found: {PRETRAINED_MODEL_PATH.resolve()}'
    )


def to_n_dhw(array, label):
    """Convert supported volume layouts to (N, 128, 256, 256)."""
    array = np.asarray(array, dtype=np.float32)
    if array.ndim != 4:
        raise ValueError(f'{label} must be 4-D; got {array.shape}')
    if array.shape[1:] == (128, 256, 256):
        return array
    if array.shape[:3] == (128, 256, 256):
        return np.transpose(array, (3, 0, 1, 2))
    if array.shape[1:] == (256, 256, 128):
        return np.transpose(array, (0, 3, 1, 2))
    if array.shape[:3] == (256, 256, 128):
        return np.transpose(array, (3, 2, 0, 1))
    raise ValueError(
        f'{label} has unsupported shape {array.shape}; expected an N/D/H/W permutation '
        'compatible with (N, 128, 256, 256).'
    )


def first_available_key(archive, candidates):
    return next((key for key in candidates if key in archive.files), None)


def load_lung_preferred_training_data():
    masked_path = next((path for path in MASKED_ARCHIVE_CANDIDATES if path.exists()), None)
    archive_path = masked_path or RAW_DATA_PATH
    if not archive_path.exists():
        raise FileNotFoundError(f'Training archive was not found: {archive_path.resolve()}')

    with np.load(archive_path, allow_pickle=False) as archive:
        volume_key = first_available_key(archive, VOLUME_KEYS)
        if volume_key is None:
            raise KeyError(f'{archive_path} has no volume key. Available keys: {archive.files}')
        volumes = to_n_dhw(archive[volume_key], f'{archive_path}:{volume_key}')

        mask_key = first_available_key(archive, MASK_KEYS)
        masks = None if mask_key is None else to_n_dhw(archive[mask_key], f'{archive_path}:{mask_key}')

    if masks is not None:
        if masks.shape != volumes.shape:
            raise ValueError(f'Volume/mask shape mismatch: {volumes.shape} vs {masks.shape}')
        masks = (masks > 0).astype(np.float32)
        volumes = volumes * masks
        print(f'Using lung mask key: {mask_key} from {archive_path}')
    elif masked_path is not None:
        print(f'Using lung-masked volume archive: {archive_path}')
    else:
        print('WARNING: no lung mask or masked-volume archive was found; using unmasked volumes.')
        print('Checked masked archives:', [str(path) for path in MASKED_ARCHIVE_CANDIDATES])

    print('Fine-tuning volume shape:', volumes.shape)
    return volumes, masks, archive_path


def different_patient_generator(volumes, masks=None, batch_size=2, seed=42):
    """Yield moving/fixed batches whose patient indices are always different."""
    if len(volumes) < 2:
        raise ValueError('At least two patients are required for different-patient fine-tuning.')
    rng = np.random.default_rng(seed)
    while True:
        moving_indices = rng.integers(0, len(volumes), size=batch_size)
        fixed_indices = rng.integers(0, len(volumes), size=batch_size)
        while np.any(moving_indices == fixed_indices):
            repeated = moving_indices == fixed_indices
            fixed_indices[repeated] = rng.integers(0, len(volumes), size=repeated.sum())

        moving = torch.from_numpy(volumes[moving_indices]).unsqueeze(1)
        fixed = torch.from_numpy(volumes[fixed_indices]).unsqueeze(1)
        if masks is None:
            overlap = None
        else:
            moving_mask = torch.from_numpy(masks[moving_indices]).unsqueeze(1)
            fixed_mask = torch.from_numpy(masks[fixed_indices]).unsqueeze(1)
            overlap = moving_mask * fixed_mask
        yield moving, fixed, overlap, moving_indices, fixed_indices


def masked_mse(target, prediction, mask=None, eps=1e-6):
    squared_error = (target - prediction).square()
    if mask is None:
        return squared_error.mean()
    return (squared_error * mask).sum() / mask.sum().clamp_min(eps)


def smoothness_loss(flow):
    dz = (flow[:, :, 1:, :, :] - flow[:, :, :-1, :, :]).square().mean()
    dy = (flow[:, :, :, 1:, :] - flow[:, :, :, :-1, :]).square().mean()
    dx = (flow[:, :, :, :, 1:] - flow[:, :, :, :, :-1]).square().mean()
    return (dx + dy + dz) / 3.0


def reconstruct_warped(moving_images, fixed_images):
    moving_analysis = analysis_filter_3d(moving_images, analysis)
    fixed_analysis = analysis_filter_3d(fixed_images, analysis)
    moving_w = down_sampling_3d(moving_analysis).to(device)
    fixed_w = down_sampling_3d(fixed_analysis).to(device)
    flow = model3D(moving_w, fixed_w)
    warped_bands = [
        transformer(moving_w[:, band:band + 1], flow)
        for band in range(moving_w.shape[1])
    ]
    warped_wavelets = torch.cat(warped_bands, dim=1)
    warped_up = up_sampling_3d(warped_wavelets)
    warped_image, filtered_bands = synthesis_filter_3d(warped_up, synthesis_filters)
    return warped_image.to(device), flow, moving_w, warped_up, filtered_bands


volumes, lung_masks, archive_path = load_lung_preferred_training_data()
finetune_generator = different_patient_generator(volumes, masks=lung_masks, batch_size=2)

model3D = vxm.networks.VxmDense_128_256_256(
    (128, 256, 256), nb_features, int_steps=0
).to(device)
try:
    checkpoint = torch.load(PRETRAINED_MODEL_PATH, map_location=device, weights_only=True)
except TypeError:  # PyTorch versions without weights_only
    checkpoint = torch.load(PRETRAINED_MODEL_PATH, map_location=device)
model3D.load_state_dict(checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint)
model3D.train()
print(f'Loaded 80k pretrained model: {PRETRAINED_MODEL_PATH.resolve()}')

transformer = vxm.layers.SpatialTransformer((64, 128, 128)).to(device)
optimizer = optim.Adam(model3D.parameters(), lr=1e-6)

finetune_epochs = 30000
smoothness_weight = 0.1
evaluation_every = 100
checkpoint_every = 1000
checkpoint_dir = Path('finetune_checkpoints_different_patients')
checkpoint_dir.mkdir(exist_ok=True)

losses, image_losses, smooth_losses = [], [], []

for epoch in tqdm(range(1, finetune_epochs + 1), desc='Fine-tuning'):
    moving_cpu, fixed_cpu, overlap_cpu, moving_ids, fixed_ids = next(finetune_generator)
    moving_images = moving_cpu.to(device=device, dtype=torch.float32)
    fixed_images = fixed_cpu.to(device=device, dtype=torch.float32)
    overlap_mask = None if overlap_cpu is None else overlap_cpu.to(device=device, dtype=torch.float32)

    optimizer.zero_grad(set_to_none=True)
    transformed_image, Vec, moving_w, moving_warped_up, filtered_bands = reconstruct_warped(
        moving_images, fixed_images
    )
    loss_image = masked_mse(fixed_images, transformed_image, overlap_mask)
    loss_smooth = smoothness_loss(Vec)
    loss = loss_image + smoothness_weight * loss_smooth
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model3D.parameters(), max_norm=1.0)
    optimizer.step()

    losses.append(loss.detach().cpu().item())
    image_losses.append(loss_image.detach().cpu().item())
    smooth_losses.append(loss_smooth.detach().cpu().item())

    if epoch % 10 == 0:
        print(
            f'Epoch {epoch}/{finetune_epochs} | loss={loss.item():.6f} | '
            f'image={loss_image.item():.6f} | smooth={loss_smooth.item():.6f} | '
            f'patients={moving_ids.tolist()} -> {fixed_ids.tolist()}'
        )

    if epoch % checkpoint_every == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model3D.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss.item(),
            'archive_path': str(archive_path),
            'used_lung_mask': lung_masks is not None,
        }, checkpoint_dir / f'finetune_epoch_{epoch:05d}.pth')

    if epoch % evaluation_every == 0:
        # Recompute after optimizer.step() so the display corresponds to the saved state.
        model3D.eval()
        with torch.no_grad():
            transformed_eval, Vec_eval, _, _, _ = reconstruct_warped(moving_images, fixed_images)
            mse_before = masked_mse(fixed_images, moving_images, overlap_mask).item()
            mse_after = masked_mse(fixed_images, transformed_eval, overlap_mask).item()
            mae_before = (
                (fixed_images - moving_images).abs().mean()
                if overlap_mask is None else
                ((fixed_images - moving_images).abs() * overlap_mask).sum() / overlap_mask.sum().clamp_min(1e-6)
            ).item()
            mae_after = (
                (fixed_images - transformed_eval).abs().mean()
                if overlap_mask is None else
                ((fixed_images - transformed_eval).abs() * overlap_mask).sum() / overlap_mask.sum().clamp_min(1e-6)
            ).item()
        model3D.train()

        print(f'\n--- Epoch {epoch} evaluation ---')
        print(f'MSE before/after: {mse_before:.6f} / {mse_after:.6f}')
        print(f'MAE before/after: {mae_before:.6f} / {mae_after:.6f}')
        print(
            f'Vec min/max/mean|abs|: {Vec_eval.min().item():.3f} / '
            f'{Vec_eval.max().item():.3f} / {Vec_eval.abs().mean().item():.6f}'
        )

        slice_idx = min(64, moving_images.shape[2] - 1)
        plt.figure(figsize=(12, 4))
        for panel, (title, image) in enumerate([
            ('Moving', moving_images[0, 0, slice_idx]),
            ('Fixed', fixed_images[0, 0, slice_idx]),
            ('Warped', transformed_eval[0, 0, slice_idx]),
        ], start=1):
            plt.subplot(1, 3, panel)
            plt.imshow(image.detach().cpu(), cmap='gray')
            plt.title(title)
            plt.axis('off')
        plt.tight_layout()
        plt.show()

        if mse_after >= mse_before:
            print('Warning: this sampled pair did not improve; inspect the displayed image and flow range.')

final_path = checkpoint_dir / 'finetuned_different_patients_final.pth'
torch.save({
    'epoch': finetune_epochs,
    'model_state_dict': model3D.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'archive_path': str(archive_path),
    'used_lung_mask': lung_masks is not None,
}, final_path)
print(f'Fine-tuned model saved: {final_path}')


In [14]:
import os

for drive in ["D:\\", "C:\\"]:
    for root, dirs, files in os.walk(drive):
        for file in files:
            if "wavelet" in file.lower() and file.endswith(".pth"):
                print(os.path.join(root, file))

D:\Saito\model_wavelet_128_256_256.pth
D:\Saito\model_wavelet_finetune_final.pth
D:\Saito\model_wavelet_pretrain_checkpoint.pth
D:\Saito\model_wavelet_pretrain_final.pth
D:\Saito\model_wavelet_pretrain_final_80000.pth
D:\Saito\model_wavelet_pretrain_restart_checkpoint.pth
D:\Saito\Saito_model_wavelet_128_256_256.pth
D:\Yamato\model_VXM_3D_MInoBed_WaveletEncorder.pth
D:\Yamato\model_VXM_3D_MInoBed_WaveletTest.pth
D:\Yamato\model_VXM_3D_weights_Wavelet.pth
C:\Users\user\OneDrive\ドキュメント\model_VXM_3D_MInoBed_WaveletTest.pth


In [15]:
import torch, os

save_path = r"D:\Saito\model_wavelet_128_256_256.pth"
torch.save(model3D.state_dict(), save_path)

print(os.path.exists(save_path))
print(save_path)

True
D:\Saito\model_wavelet_128_256_256.pth


In [ ]:
# Check_Perfect_Reconstruction.py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt

pr_names = ['LLL', 'LLH', 'LHL', 'LHH', 'HLL', 'HLH', 'HHL', 'HHH']

class CheckPRHaar3DAnalysis(nn.Module):
    def __init__(self):
        super().__init__()
        hL = torch.tensor([1.0, 1.0], dtype=torch.float32) / math.sqrt(2)
        hH = torch.tensor([1.0, -1.0], dtype=torch.float32) / math.sqrt(2)

        filters = []
        filter_names = []
        for z_name, z_filter in zip(['L', 'H'], [hL, hH]):
            for y_name, y_filter in zip(['L', 'H'], [hL, hH]):
                for x_name, x_filter in zip(['L', 'H'], [hL, hH]):
                    kernel = (
                        z_filter[:, None, None]
                        * y_filter[None, :, None]
                        * x_filter[None, None, :]
                    )
                    filters.append(kernel)
                    filter_names.append(z_name + y_name + x_name)

        weight = torch.stack(filters, dim=0).unsqueeze(1)
        self.register_buffer('weight', weight)
        self.names = filter_names

    def forward(self, x):
        x_pad = F.pad(x, (0, 1, 0, 1, 0, 1))
        w = F.conv3d(x_pad, self.weight, stride=1, padding=0)
        return w

def check_pr_downsample_3d(w):
    return w[:, :, ::2, ::2, ::2]

def check_pr_upsample_3d(w_down):
    B, C, D, H, W = w_down.shape
    w_up = torch.zeros(B, C, D * 2, H * 2, W * 2, dtype=w_down.dtype, device=w_down.device)
    w_up[:, :, ::2, ::2, ::2] = w_down
    return w_up

def check_pr_make_3d_filter(fz, fy, fx):
    return fz[:, None, None] * fy[None, :, None] * fx[None, None, :]

class CheckPRHaar3DSynthesis(nn.Module):
    def __init__(self):
        super().__init__()
        sqrt2 = math.sqrt(2.0)
        low = torch.tensor([1.0, 1.0], dtype=torch.float32) / sqrt2
        high = torch.tensor([1.0, -1.0], dtype=torch.float32) / sqrt2

        filters = torch.stack([
            check_pr_make_3d_filter(low, low, low),
            check_pr_make_3d_filter(low, low, high),
            check_pr_make_3d_filter(low, high, low),
            check_pr_make_3d_filter(low, high, high),
            check_pr_make_3d_filter(high, low, low),
            check_pr_make_3d_filter(high, low, high),
            check_pr_make_3d_filter(high, high, low),
            check_pr_make_3d_filter(high, high, high),
        ], dim=0)

        filters = torch.flip(filters, dims=[1, 2, 3]).unsqueeze(1)
        self.register_buffer('filters', filters)

    def forward(self, w_up):
        B, C, D, H, W = w_up.shape
        filtered_bands = []
        for i, name in enumerate(pr_names):
            band = w_up[:, i:i + 1, :, :, :]
            kernel = self.filters[i:i + 1]
            filtered = F.conv3d(band, kernel, stride=1, padding=1)
            filtered = filtered[:, :, :D, :H, :W]
            filtered_bands.append(filtered)
            print(name, 'Synthesis後:', filtered.shape)

        filtered_bands = torch.cat(filtered_bands, dim=1)
        reconstructed = torch.sum(filtered_bands, dim=1, keepdim=True)
        return filtered_bands, reconstructed

def check_pr_frequency_response(h, omega):
    n = np.arange(len(h))
    response = np.sum(h[None, :] * np.exp(-1j * omega[:, None] * n[None, :]), axis=1)
    return response

In [35]:
# Check_Perfect_Reconstruction.py: reconstruction check
check_pr_x = torch.as_tensor(
    x_train[0:1],
    dtype=torch.float32,
    device=device
).unsqueeze(1)

print('\n===================================')
print('入力')
print('===================================')
print('元画像:', check_pr_x.shape)

print('\n===================================')
print('1. Analysis Filter')
print('===================================')
check_pr_analysis = CheckPRHaar3DAnalysis().to(device)
check_pr_w = check_pr_analysis(check_pr_x)
print('Analysis後:', check_pr_w.shape)
print('周波数成分:', check_pr_analysis.names)

print('\n===================================')
print('2. Downsampling')
print('===================================')
check_pr_w_down = check_pr_downsample_3d(check_pr_w)
print('Downsampling後:', check_pr_w_down.shape)

print('\n===================================')
print('3. Upsampling')
print('===================================')
check_pr_w_up = check_pr_upsample_3d(check_pr_w_down)
print('Upsampling後:', check_pr_w_up.shape)

check_pr_up_error = torch.abs(check_pr_w_up[:, :, ::2, ::2, ::2] - check_pr_w_down)
print('Upsampling配置確認 平均誤差:', check_pr_up_error.mean().item())
print('Upsampling配置確認 最大誤差:', check_pr_up_error.max().item())

print('\n===================================')
print('4. Synthesis Filter')
print('===================================')
check_pr_synthesis = CheckPRHaar3DSynthesis().to(device)
check_pr_filtered_bands, check_pr_reconstructed = check_pr_synthesis(check_pr_w_up)
print('\nSynthesis後8成分:', check_pr_filtered_bands.shape)
print('再構成画像:', check_pr_reconstructed.shape)

print('\n===================================')
print('5. Reconstruction Error')
print('===================================')

if check_pr_x.shape != check_pr_reconstructed.shape:
    print('サイズが一致していません')
    print('Original:', check_pr_x.shape)
    print('Reconstructed:', check_pr_reconstructed.shape)
else:
    print('サイズ一致')
    check_pr_diff = check_pr_x - check_pr_reconstructed
    check_pr_absolute_error = torch.abs(check_pr_diff)
    check_pr_mae = torch.mean(check_pr_absolute_error)
    check_pr_max_error = torch.max(check_pr_absolute_error)
    check_pr_mse = torch.mean(check_pr_diff ** 2)
    check_pr_relative_error = torch.norm(check_pr_diff) / torch.norm(check_pr_x)
    print('\n===== 再構成誤差 =====')
    print('MAE:', check_pr_mae.item())
    print('最大絶対誤差:', check_pr_max_error.item())
    print('MSE:', check_pr_mse.item())
    print('相対誤差:', check_pr_relative_error.item())

    if check_pr_max_error.item() > 0:
         check_pr_error_order = int(np.floor(np.log10(check_pr_max_error.item())))
         check_pr_error_scale = 10.0 ** (-check_pr_error_order)
    else:
         check_pr_error_order = 0
         check_pr_error_scale = 1.0

    print('誤差表示スケール:', f'x 1e{-check_pr_error_order}' if check_pr_max_error.item() > 0 else 'no scaling')

print('\n===================================')
print('6. Save Results')
print('===================================')
np.save(r'D:\Saito\wavelet_reconstructed.npy', check_pr_reconstructed.detach().cpu().numpy())
print('再構成画像を保存しました')

if check_pr_x.shape == check_pr_reconstructed.shape:
    check_pr_slice_index = check_pr_x.shape[2] // 2

    plt.figure(figsize=(8, 8))
    plt.imshow(check_pr_x[0, 0, check_pr_slice_index].detach().cpu().numpy(), cmap='gray')
    plt.title('Original Image')
    plt.axis('off')
    plt.savefig(r'D:\Saito\wavelet_original.png', dpi=300, bbox_inches='tight')
    plt.close()

    plt.figure(figsize=(8, 8))
    plt.imshow(check_pr_reconstructed[0, 0, check_pr_slice_index].detach().cpu().numpy(), cmap='gray')
    plt.title('Reconstructed Image')
    plt.axis('off')
    plt.savefig(r'D:\Saito\wavelet_reconstructed.png', dpi=300, bbox_inches='tight')
    plt.close()

    plt.figure(figsize=(8, 8))
    plt.imshow(
         check_pr_absolute_error[0, 0, check_pr_slice_index].detach().cpu().numpy(),
         cmap='inferno',
         vmin=0.0,
         vmax=check_pr_max_error.item()
     )
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.title('Absolute Reconstruction Error')
    plt.axis('off')
    plt.savefig(r'D:\Saito\wavelet_error.png', dpi=300, bbox_inches='tight')
    plt.close()

    plt.figure(figsize=(8, 8))
    plt.imshow(
       (check_pr_absolute_error[0, 0, check_pr_slice_index] * check_pr_error_scale).detach().cpu().numpy(),
       cmap='inferno'
     )
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.title(
         f'Scaled Error (x 1e{-check_pr_error_order})'
         if check_pr_max_error.item() > 0
         else 'Scaled Error'
     )
    plt.axis('off')
    plt.savefig(r'D:\Saito\wavelet_error_scaled.png', dpi=300, bbox_inches='tight')
    plt.close()


入力
元画像: torch.Size([1, 1, 128, 256, 256])

1. Analysis Filter
Analysis後: torch.Size([1, 8, 128, 256, 256])
周波数成分: ['LLL', 'LLH', 'LHL', 'LHH', 'HLL', 'HLH', 'HHL', 'HHH']

2. Downsampling
Downsampling後: torch.Size([1, 8, 64, 128, 128])

3. Upsampling
Upsampling後: torch.Size([1, 8, 128, 256, 256])
Upsampling配置確認 平均誤差: 0.0
Upsampling配置確認 最大誤差: 0.0

4. Synthesis Filter
LLL Synthesis後: torch.Size([1, 1, 128, 256, 256])
LLH Synthesis後: torch.Size([1, 1, 128, 256, 256])
LHL Synthesis後: torch.Size([1, 1, 128, 256, 256])
LHH Synthesis後: torch.Size([1, 1, 128, 256, 256])
HLL Synthesis後: torch.Size([1, 1, 128, 256, 256])
HLH Synthesis後: torch.Size([1, 1, 128, 256, 256])
HHL Synthesis後: torch.Size([1, 1, 128, 256, 256])
HHH Synthesis後: torch.Size([1, 1, 128, 256, 256])

Synthesis後8成分: torch.Size([1, 8, 128, 256, 256])
再構成画像: torch.Size([1, 1, 128, 256, 256])

5. Reconstruction Error
サイズ一致

===== 再構成誤差 =====
MAE: 3.967887707290174e-08
最大絶対誤差: 4.172325134277344e-07
MSE: 5.188463472473011e-15
相対誤差:

In [36]:
if check_pr_x.shape == check_pr_reconstructed.shape:
    check_pr_slice_index = check_pr_x.shape[2] // 2

    # ========================================
    # 元画像
    # ========================================
    plt.figure(figsize=(8, 8))

    plt.imshow(
        check_pr_x[
            0,
            0,
            check_pr_slice_index
        ].detach().cpu().numpy(),
        cmap='gray'
    )

    plt.title('Original Image')
    plt.axis('off')

    plt.savefig(
        r'D:\Saito\wavelet_original.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()


    # ========================================
    # 再構成画像
    # ========================================
    plt.figure(figsize=(8, 8))

    plt.imshow(
        check_pr_reconstructed[
            0,
            0,
            check_pr_slice_index
        ].detach().cpu().numpy(),
        cmap='gray'
    )

    plt.title('Reconstructed Image')
    plt.axis('off')

    plt.savefig(
        r'D:\Saito\wavelet_reconstructed.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()


    # ========================================
    # 絶対誤差画像
    # ========================================
    plt.figure(figsize=(8, 8))

    plt.imshow(
        check_pr_absolute_error[
            0,
            0,
            check_pr_slice_index
        ].detach().cpu().numpy(),
        cmap='inferno',
        vmin=0.0,
        vmax=check_pr_max_error.item()
    )

    plt.colorbar(
        fraction=0.046,
        pad=0.04
    )

    plt.title('Absolute Reconstruction Error')
    plt.axis('off')

    plt.savefig(
        r'D:\Saito\wavelet_error.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()


    # ========================================
    # 拡大表示した誤差画像
    # ========================================
    plt.figure(figsize=(8, 8))

    check_pr_scaled_error = (
        check_pr_absolute_error[
            0,
            0,
            check_pr_slice_index
        ]
        * check_pr_error_scale
    ).detach().cpu().numpy()

    plt.imshow(
        check_pr_scaled_error,
        cmap='inferno'
    )

    plt.colorbar(
        fraction=0.046,
        pad=0.04
    )

    if check_pr_max_error.item() > 0:
        plt.title(
            f'Scaled Error (x 1e{-check_pr_error_order})'
        )
    else:
        plt.title('Scaled Error')

    plt.axis('off')

    plt.savefig(
        r'D:\Saito\wavelet_error_scaled.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

    print('画像を保存しました')

画像を保存しました


In [37]:
# Check_Perfect_Reconstruction.py: perfect reconstruction conditions
print('\n===================================')
print('7. Perfect Reconstruction Conditions')
print('===================================')

check_pr_sqrt2 = np.sqrt(2.0)
check_pr_h0 = np.array([1.0, 1.0]) / check_pr_sqrt2
check_pr_h1 = np.array([-1.0, 1.0]) / check_pr_sqrt2
check_pr_f0 = np.array([1.0, 1.0]) / check_pr_sqrt2
check_pr_f1 = np.array([1.0, -1.0]) / check_pr_sqrt2

check_pr_N = 2048
check_pr_omega = np.linspace(-np.pi, np.pi, check_pr_N, endpoint=False)

check_pr_H0 = check_pr_frequency_response(check_pr_h0, check_pr_omega)
check_pr_H1 = check_pr_frequency_response(check_pr_h1, check_pr_omega)
check_pr_F0 = check_pr_frequency_response(check_pr_f0, check_pr_omega)
check_pr_F1 = check_pr_frequency_response(check_pr_f1, check_pr_omega)
check_pr_H0_minus = check_pr_frequency_response(check_pr_h0, check_pr_omega + np.pi)
check_pr_H1_minus = check_pr_frequency_response(check_pr_h1, check_pr_omega + np.pi)

check_pr_alias_term = check_pr_H0_minus * check_pr_F0 + check_pr_H1_minus * check_pr_F1
check_pr_alias_max = np.max(np.abs(check_pr_alias_term))
print('\n-----------------------------------')
print('条件1: Alias Cancellation')
print('-----------------------------------')
print('最大Alias成分:', check_pr_alias_max)

check_pr_T = check_pr_H0 * check_pr_F0 + check_pr_H1 * check_pr_F1
check_pr_magnitude = np.abs(check_pr_T)
check_pr_magnitude_min = np.min(check_pr_magnitude)
check_pr_magnitude_max = np.max(check_pr_magnitude)
check_pr_magnitude_variation = check_pr_magnitude_max - check_pr_magnitude_min
print('\n-----------------------------------')
print('条件2: Amplitude Distortion')
print('-----------------------------------')
print('Magnitude min:', check_pr_magnitude_min)
print('Magnitude max:', check_pr_magnitude_max)
print('Magnitude variation:', check_pr_magnitude_variation)

check_pr_phase = np.unwrap(np.angle(check_pr_T))
check_pr_phase_coef = np.polyfit(check_pr_omega, check_pr_phase, 1)
check_pr_phase_fit = np.polyval(check_pr_phase_coef, check_pr_omega)
check_pr_phase_error = check_pr_phase - check_pr_phase_fit
check_pr_max_phase_error = np.max(np.abs(check_pr_phase_error))
print('\n-----------------------------------')
print('条件3: Phase Distortion')
print('-----------------------------------')
print('Phase slope:', check_pr_phase_coef[0])
print('最大直線位相誤差:', check_pr_max_phase_error)

check_pr_tolerance = 1e-10
print('\n===================================')
print('PR Condition Results')
print('===================================')
print('条件1 Alias Cancellation:', 'OK' if check_pr_alias_max < check_pr_tolerance else 'NG')
print('条件2 Amplitude Distortion:', 'OK' if check_pr_magnitude_variation < check_pr_tolerance else 'NG')
print('条件3 Phase Distortion:', 'OK' if check_pr_max_phase_error < check_pr_tolerance else 'NG')

plt.figure(figsize=(8, 5))
plt.plot(check_pr_omega, np.abs(check_pr_alias_term))
plt.xlabel('Angular Frequency ω')
plt.ylabel('|Alias Term|')
plt.title('Alias Cancellation')
plt.grid()
plt.savefig(r'D:\Saito\pr_alias.png', dpi=300, bbox_inches='tight')
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(check_pr_omega, check_pr_magnitude)
plt.xlabel('Angular Frequency ω')
plt.ylabel('|T(e^jω)|')
plt.title('Distortion Transfer Function Magnitude')
plt.grid()
plt.savefig(r'D:\Saito\pr_amplitude.png', dpi=300, bbox_inches='tight')
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(check_pr_omega, check_pr_phase, label='Actual Phase')
plt.plot(check_pr_omega, check_pr_phase_fit, linestyle='--', label='Linear Fit')
plt.xlabel('Angular Frequency ω')
plt.ylabel('Phase [rad]')
plt.title('Phase Response')
plt.legend()
plt.grid()
plt.savefig(r'D:\Saito\pr_phase.png', dpi=300, bbox_inches='tight')
plt.close()

print('\nPR条件確認用グラフを保存しました')
print('\n処理完了')


7. Perfect Reconstruction Conditions

-----------------------------------
条件1: Alias Cancellation
-----------------------------------
最大Alias成分: 3.554447978966673e-16

-----------------------------------
条件2: Amplitude Distortion
-----------------------------------
Magnitude min: 1.999999999999999
Magnitude max: 2.0000000000000004
Magnitude variation: 1.5543122344752192e-15

-----------------------------------
条件3: Phase Distortion
-----------------------------------
Phase slope: -1.0
最大直線位相誤差: 8.881784197001252e-16

PR Condition Results
条件1 Alias Cancellation: OK
条件2 Amplitude Distortion: OK
条件3 Phase Distortion: OK

PR条件確認用グラフを保存しました

処理完了


The fine-tuning cell above saves recoverable checkpoints every 1,000 epochs and evaluates/visualizes every 100 epochs.


In [ ]:
# チェックポイント保存は上の3万epochセルで100epochごとに実行されます。

In [ ]:
# 最終モデル保存は上の3万epochセルの完了時に実行されます。